In [3]:
from typing import Annotated, Optional, List
from typing_extensions import TypedDict
import os 
import asyncio

# openAI chatModel
from langchain_openai import ChatOpenAI

from langchain_core.tools import tool
from langchain_core.tools import StructuredTool

# for graph api 
from langgraph.graph import StateGraph , START, END, add_messages

# for tooling
from langgraph.prebuilt import ToolNode, tools_condition

# for message stuff
from langchain_core.messages import SystemMessage, ToolMessage, HumanMessage

from IPython.display import Image, display

from dotenv import load_dotenv
load_dotenv()

True

In [4]:
llm = ChatOpenAI(
    model=os.getenv("MODEL_ID"),
    base_url = os.getenv("OPENROUTER_SERVER")   
)
print(llm.model_name)

response =llm.invoke("what is your purpose")
print(response.content)

deepseek/deepseek-v4-flash
That's a very fundamental question, and I appreciate you asking it.

In simple terms, my purpose is to be **a helpful and harmless AI assistant.**

But let's break that down a bit, because it's more than just a tagline.

My core purpose, the reason I was created, is to **serve your needs.** This means I'm designed to help you with a wide range of tasks, including:

- **Answering questions:** From "What's the capital of Mongolia?" to "Explain the theory of general relativity."
- **Generating content:** Writing emails, stories, poems, code, scripts, musical pieces, and more.
- **Summarizing and analyzing:** Breaking down long documents, articles, or conversations into key points.
- **Brainstorming and problem-solving:** Helping you think through a challenge, come up with ideas, or plan a project.
- **Learning and exploration:** Explaining complex topics, teaching you a new skill, or translating languages.
- **Being a conversational partner:** Discussing ideas, 

In [5]:
SYSTEM_PROMPT = SystemMessage(
    content=(
        "You are a calculator agent. You must NEVER perform arithmetic yourself, "
        "not even trivial arithmetic. Every single operation goes through a tool "
        "call, one operation at a time, respecting the normal order of operations. "
        "If a tool returns an error, read it and adjust your approach. "
        "When you have the final number, state it plainly."
    )
)


In [6]:
# state schema 

class CalculatorSchema(TypedDict) :
    messages : Annotated[list, add_messages]

In [ ]:
@tool
def add(a: float, b: float) -> float:
    """Adds two numbers together."""
    return a + b

@tool
def subtract(a: float, b: float) -> float:
    """Subtracts the second number from the first."""
    return a - b

@tool
def multiply(a: float, b: float) -> float:
    """Multiplies two numbers."""
    return a * b

@tool
def divide(a: float, b: float) -> float:
    """Divides the first number by the second."""
    if b == 0:
        return "Error: Cannot divide by zero."
    return a / b

@tool
def user_input(query):
    """_summary_

    Args:
        query (_type_): _description_
    """
    user_input = input(query)
    return user_input

tools: Optional[List[StructuredTool]] = [
    add ,
    subtract,
    multiply,
    divide
]
# tools = asyncio.run(client.get_tools())
print([f.name for f in tools])


llm_with_tools = llm.bind_tools(tools)

['add', 'subtract', 'multiply', 'divide']


In [ ]:
# node defination 

def call_agent(state: CalculatorSchema) -> CalculatorSchema:
    """Invokes the LLM with the current conversation state."""
    response = llm_with_tools.invoke([SYSTEM_PROMPT] + state["messages"])
    
    return {"messages": [response]}

def modified_call_agent(state: CalculatorSchema):
    """Invokes the LLM with the current conversation state."""
    messages = state["messages"]
    response = llm_with_tools.invoke(messages) 
    
    ## the only difference is System_Prompt is injected in above with state[messages] !!!!
    ## response = llm_with_tools.invoke([SYSTEM_PROMPT]+state[messages]) 
    
    # Return a dictionary updating the state with the new message
    return {"messages": [response]}

tool_node = ToolNode(tools)
    

In [ ]:
# define the graph

#initialize state graph
workflow = StateGraph(CalculatorSchema)

# add node to the graph
workflow.add_node("call_agent", call_agent)
workflow.add_node("tool_calling_node", tool_node) 

# define the edges 
workflow.add_edge(START,"call_agent")
workflow.add_conditional_edges(
        "call_agent",
        tools_condition,
        {
            "tools": "tool_calling_node", # Maps the router's "tools" path to your custom node name
            "__end__": "__end__"          # Keeps the end path intact
        }
    )
workflow.add_edge("tool_calling_node", "call_agent")

app = workflow.compile()

display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# 1. Run the graph using invoke() instead of stream()
# This blocks until the graph reaches END and returns the final state dictionary
final_state = app.invoke({"messages": [HumanMessage(content="what is the theory of relativity ?")]})

# 2. Extract the messages array
all_messages = final_state["messages"]

# 3. Iterate and format the output
print("=== FULL STATE HISTORY ===\n")

for i, msg in enumerate(all_messages):
    # Get the type of message (HumanMessage, AIMessage, ToolMessage)
    msg_type = msg.__class__.__name__
    
    print(f"[{i + 1}] {msg_type}")
    
    # Print the text content if it exists
    if msg.content:
        print(f"Text: {msg.content}")
    
    # If the AI made a tool call, the content is often empty, but the tool_calls array is populated
    if isinstance(msg, AIMessage) and msg.tool_calls:
        for call in msg.tool_calls:
            print(f"Action: Called '{call['name']}' with arguments {call['args']}")
            
    # If it's a ToolMessage, print the result and the ID it responded to
    if isinstance(msg, ToolMessage):
        print(f"Result: {msg.content}")
        print(f"(Responded to tool_call_id: {msg.tool_call_id})")
        
    print("-" * 40)